# RNN: hàm, trạng thái ẩn và BPTT

Nguyễn Ngọc Hoàng Nam — B23DCCN585 | ASG 06

Các cell nhỏ được sắp theo thứ tự phụ thuộc. Huấn luyện đầy đủ mặc định tắt; số liệu chạy thử không phải kết quả test chính thức.

In [1]:
from pathlib import Path
import sys, json, os
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks": ROOT=ROOT.parent
assert (ROOT/"src").is_dir()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.paths import DATA
from src.io import read_json
import numpy as np
import pandas as pd
from IPython.display import display, Image
print("Project:", ROOT)
print("Data:", DATA)


Project: E:\PTHTTM\ASG_06
Data: E:\PTHTTM\ASG_06_data


## 1. Phép truy hồi
$h_t=\tanh(x_tW_x+h_{t-1}W_h+b)$. Cùng bộ trọng số được dùng ở mọi bước thời gian.

In [2]:
def rnn_forward(x, wx, wh, bias):
    h=np.zeros(wh.shape[0],dtype="float64")
    states=[h.copy()]
    for xt in x:
        h=np.tanh(xt@wx+h@wh+bias)
        states.append(h.copy())
    return np.asarray(states)

In [3]:
x=np.array([[1.,0.],[0.,1.],[1.,1.]])
wx=np.array([[.3,-.2],[.1,.4]])
wh=np.eye(2)*.5
b=np.zeros(2)
states=rnn_forward(x,wx,wh,b)
display(pd.DataFrame(states,columns=['h1','h2']))

,h1,h2
0,0.000000,0.000000
1,0.291313,-0.197375
2,0.240831,0.292513
3,0.478021,0.333052


## 2. Thứ tự có ý nghĩa
Đảo thứ tự đầu vào có thể đổi trạng thái cuối dù tập giá trị không đổi.

In [4]:
print('Ban đầu:',states[-1])
print('Đảo thứ tự:',rnn_forward(x[::-1],wx,wh,b)[-1])

Ban đầu: [0.47802071 0.3330515 ]
Đảo thứ tự: [0.4145191  0.03053273]


## 3. Lan truyền ngược qua thời gian
Gradient của trọng số dùng chung là tổng đóng góp của từng bước.

In [5]:
def last_state_gradient(x,wx,wh,bias,target):
    states=rnn_forward(x,wx,wh,bias)
    loss=0.5*np.sum((states[-1]-target)**2)
    dh=states[-1]-target
    dwx=np.zeros_like(wx);dwh=np.zeros_like(wh);db=np.zeros_like(bias)
    for t in range(len(x)-1,-1,-1):
        dz=dh*(1-states[t+1]**2)
        dwx+=np.outer(x[t],dz)
        dwh+=np.outer(states[t],dz)
        db+=dz
        dh=dz@wh.T
    return loss,(dwx,dwh,db)

In [6]:
def finite_difference(function, array, epsilon=1e-6):
    gradient=np.zeros_like(array)
    for index in np.ndindex(array.shape):
        plus=array.copy();minus=array.copy()
        plus[index]+=epsilon;minus[index]-=epsilon
        gradient[index]=(function(plus)-function(minus))/(2*epsilon)
    return gradient

In [7]:
target=np.array([.2,.1])
loss,grads=last_state_gradient(x,wx,wh,b,target)
numeric=finite_difference(lambda a:last_state_gradient(x,a,wh,b,target)[0],wx)
np.testing.assert_allclose(grads[0],numeric,atol=1e-7)
print('Loss:',loss,'Max gradient error:',np.abs(grads[0]-numeric).max())

Loss: 0.06580425875839721 Max gradient error: 9.064582418005784e-12


## 4. Độ dài chuỗi và gradient
Tích nhiều hệ số nhỏ có thể làm gradient tiêu biến; gradient clipping chỉ giới hạn gradient quá lớn.

In [8]:
steps=np.arange(1,31)
display(pd.DataFrame({'steps':steps,'0.5^steps':.5**steps,'1.5^steps':1.5**steps}))

,steps,0.5^steps,1.5^steps
0,1,5.000000e-01,1.500000
1,2,2.500000e-01,2.250000
2,3,1.250000e-01,3.375000
3,4,6.250000e-02,5.062500
4,5,3.125000e-02,7.593750
5,6,1.562500e-02,11.390625
6,7,7.812500e-03,17.085938
7,8,3.906250e-03,25.628906
8,9,1.953125e-03,38.443359
9,10,9.765625e-04,57.665039
